### Imports

In [10]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import ollama
from neo4j import GraphDatabase
import re
import json
import chromadb

### Scraping Testing

In [3]:
def scrape_website(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=10)
    
    print("status:", response.status_code)
    print("raw html length:", len(response.text))
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator="\n", strip=True)
    return text

url = "https://www.slt.lk"
text = scrape_website(url)
print(len(text))
print(text[:5000])

status: 200
raw html length: 4032
265
SLTMobitel | Broadband, TV, Telephone and Business solutions
Mobile
Postpaid & prepaid connections, Broadband ( 3G & 4G/LTE), Digital Services, MCash and Business Solutions.
Fixed
Broadband (Fibre, ADSL & 4G/LTE), PEOTV, SME/Micro Business and Enterprise Solutions.


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_with_selenium(url):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # no browser window
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)  # wait for js to load
    
    soup = BeautifulSoup(driver.page_source, "html.parser")
    driver.quit()
    
    for tag in soup(["script", "style"]):
        tag.decompose()
    
    text = soup.get_text(separator="\n", strip=True)
    return text

url = "https://www.slt.lk"
text = scrape_with_selenium(url)
print(len(text))
print(text[:5000])

265
SLTMobitel | Broadband, TV, Telephone and Business solutions
Mobile
Postpaid & prepaid connections, Broadband ( 3G & 4G/LTE), Digital Services, MCash and Business Solutions.
Fixed
Broadband (Fibre, ADSL & 4G/LTE), PEOTV, SME/Micro Business and Enterprise Solutions.


In [5]:
url = "https://www.dialog.lk"
text = scrape_with_selenium(url)
print(len(text))
print(text[:5000])

9871
Dialog Axiata PLC | Sri Lanka's Leading Telecommunication Company
Skip to main content
Icon/Menu
Icon/Menu
Reload & Pay
Main navigation
Mobile
Home Broadband
Television
Shop
Business
Support
Mobile
Home Broadband
Television
Shop
Business
Support
English
English
Sinhala
Tamil
English
Sinhala
Tamil
Reload & Pay
Mega menu
Mobile
Mobile Postpaid
Mobile Prepaid
Mobile Broadband
International Roaming
IDD
Tourist
Value Added Services
eSIM
Home Broadband
Postpaid
Prepaid
Fibre
Data Add-ons
Order Connection
Smart Home
Voice Plans
Television
Postpaid
Prepaid
Channel List
Order Connection
Devices
Dialog Play Hub
Dialog Play Mini
Value Added Services
Shop
Phones
Accessories
Tablets
Smart Home
About us
Overview
Governance
Investors
Structure and Management
Sustainability
Suppliers
News and Media
Global
Useful Links
Loyalty
Club Vision
Star Rewards
Star Points
Offers and Promotions
Tips to Protect from cyber scam
Digital Partners
Dialog Apps
Technology and Beyond
Digi Wasana
Support
Contact Us


### LLM Model Selection

Current model: **Gemma3:1B** — fast, clean JSON output, works well on CPU-only.

Models tested:
- **Phi3:Mini** — too slow, RAM issues on this machine

Candidates to benchmark:
- **Qwen2.5:1.5B** — known for strong JSON output
- **TinyLlama:1.1B** — fastest option, lower quality tradeoff

Note: Google Colab is an option for GPU access, but loses the fully local design goal.

In [6]:
def extract_business_info_gemma3_1b(text):
    prompt = f"""
Extract business information from the text below and return ONLY a valid JSON object.
Strict rules:
- No comments
- No trailing commas
- Use null for missing fields
- All strings must be quoted
- phone_numbers and emails must be proper lists of strings

Fields to extract:
- company_name
- address
- phone_numbers
- emails
- products_services
- website_description

Text:
{text[:3000]}

Return only the JSON object, nothing else.
"""
    response = ollama.chat(
        model="gemma3:1b",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["message"]["content"]

result = extract_business_info_gemma3_1b(text)
print("This is using the model Gemma3:1b")
print("")
print(result)

This is using the model Gemma3:1b

```json
{
  "company_name": "Dialog Axiata PLC",
  "address": null,
  "phone_numbers": [
    "07710 999999",
    "01484 444444",
    "01771 999999"
  ],
  "emails": [
    "support@dialog.co.uk",
    "info@dialog.co.uk"
  ],
  "products_services": [
    "Telecommunication",
    "Broadband",
    "TV",
    "Mobile",
    "Home Broadband",
    "Smart Home",
    "Voice Plans",
    "Data Add-on",
    "SIM",
    "Dialog TV",
    "Mobile Broadband",
    "Mobile Prepaid",
    "Mobile Broadband",
    "Internet"
  ],
  "website_description": "Dialog Axiata PLC provides telecommunications services including broadband, TV, mobile, and smart home solutions.  They offer a wide range of products and services to customers across Sri Lanka."
}
```


In [7]:
def parse_llm_output(raw):
    # remove single line comments
    raw = re.sub(r'//.*', '', raw)
    # extract json block
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError as e:
            print("json parse error:", e)
            print("raw output:", raw)
            return None
    return None

parsed = parse_llm_output(result)
print(parsed)

{'company_name': 'Dialog Axiata PLC', 'address': None, 'phone_numbers': ['07710 999999', '01484 444444', '01771 999999'], 'emails': ['support@dialog.co.uk', 'info@dialog.co.uk'], 'products_services': ['Telecommunication', 'Broadband', 'TV', 'Mobile', 'Home Broadband', 'Smart Home', 'Voice Plans', 'Data Add-on', 'SIM', 'Dialog TV', 'Mobile Broadband', 'Mobile Prepaid', 'Mobile Broadband', 'Internet'], 'website_description': 'Dialog Axiata PLC provides telecommunications services including broadband, TV, mobile, and smart home solutions.  They offer a wide range of products and services to customers across Sri Lanka.'}


### Neo4j

In [1]:
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "onetwothree"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# test connection
driver.verify_connectivity()
print("connected to neo4j")

NameError: name 'GraphDatabase' is not defined

In [9]:
def store_in_neo4j(data, url):
    with driver.session(database="lkinsight") as session:
        session.run("""
            MERGE (c:Company {name: $name})
            SET c.url = $url,
                c.address = $address,
                c.description = $description
        """, name=data["company_name"], url=url, 
             address=data["address"], 
             description=data["website_description"])
        
        for phone in data.get("phone_numbers", []):
            if phone:
                session.run("""
                    MATCH (c:Company {name: $name})
                    MERGE (p:Phone {number: $phone})
                    MERGE (c)-[:HAS_PHONE]->(p)
                """, name=data["company_name"], phone=phone)
        
        for email in data.get("emails", []):
            if email:
                session.run("""
                    MATCH (c:Company {name: $name})
                    MERGE (e:Email {address: $email})
                    MERGE (c)-[:HAS_EMAIL]->(e)
                """, name=data["company_name"], email=email)

        print("stored successfully")

store_in_neo4j(parsed, url)

stored successfully


### ChromaDB

In [11]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="lkinsight")
print("chromadb ready")

chromadb ready


In [13]:
def store_in_chroma(data, url):
    document = f"""
    Company: {data['company_name']}
    Address: {data['address']}
    Products/Services: {', '.join(data['products_services']) if isinstance(data['products_services'], list) else str(data['products_services'])}
    Description: {data['website_description']}
    """
    
    collection.add(
        documents=[document],
        ids=[url],
        metadatas=[{"company_name": data['company_name'], "url": url}]
    )
    print("stored in chromadb")

store_in_chroma(parsed, url)

stored in chromadb


In [14]:
results = collection.query(
    query_texts=["telecom company in Sri Lanka"],
    n_results=1
)
print(results)

{'ids': [['https://www.dialog.lk']], 'embeddings': None, 'documents': [['\n    Company: Dialog Axiata PLC\n    Address: None\n    Products/Services: Telecommunication, Broadband, TV, Mobile, Home Broadband, Smart Home, Voice Plans, Data Add-on, SIM, Dialog TV, Mobile Broadband, Mobile Prepaid, Mobile Broadband, Internet\n    Description: Dialog Axiata PLC provides telecommunications services including broadband, TV, mobile, and smart home solutions.  They offer a wide range of products and services to customers across Sri Lanka.\n    ']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'url': 'https://www.dialog.lk', 'company_name': 'Dialog Axiata PLC'}]], 'distances': [[1.0185918807983398]]}
